In [ ]:
pip install tiktoken

In [ ]:
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 43.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os
import json
import ast
import spacy
import numpy as np
import time
import random
import tiktoken
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from typing import List, Dict, Tuple
import statistics

# --- Configuration ---
api_key  = os.environ.get("OPENAI_API_KEY", "")
BASE_URL = None

TRAIN_DATA_PATH = "/content/train.txt"
TEST_DATA_PATH  = "/content/test.txt"
NUM_SAMPLES     = 100
NUM_FEW_SHOTS   = 1
RANDOM_SEED     = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ==========================================
# DATA LOADING
# ==========================================
def read_line_examples_from_file(data_path: str) -> Tuple[List[str], List[list]]:
    sents, labels = [], []
    with open(data_path, 'r', encoding='UTF-8') as fp:
        for line in fp:
            line = line.strip()
            if line and '####' in line:
                words, tuples_str = line.split('####', 1)
                try:
                    tuples = ast.literal_eval(tuples_str)
                    sents.append(words)
                    labels.append(tuples)
                except (ValueError, SyntaxError):
                    pass
    return sents, labels


# ==========================================
# INFERENCE TIMER CLASS
# ==========================================
class InferenceTimer:
    def __init__(self, api_key: str, base_url: str = None):
        if base_url:
            self.client = OpenAI(api_key=api_key, base_url=base_url)
        else:
            self.client = OpenAI(api_key=api_key)
        self.nlp_sm = spacy.load("en_core_web_sm")
        self.nlp_lg = spacy.load("en_core_web_lg")
        self.enc    = tiktoken.encoding_for_model("gpt-4o-mini")

    def time_dependency_parsing(self, sentence: str) -> Tuple[float, str]:
        """Measures dependency parsing time. Returns (time, dependency_str)."""
        start = time.perf_counter()
        doc = self.nlp_sm(sentence)
        dep_lines = [
            f"{token.text}-{token.head.text}->{token.dep_}"
            for token in doc
        ]
        dependency_str = "\n".join(dep_lines)
        end = time.perf_counter()
        return end - start, dependency_str

    def time_retrieval(
        self,
        sentence: str,
        train_embeddings: np.ndarray,
        train_sents: List[str],
        train_labels: List[list],
        n: int = 1
    ) -> Tuple[float, List[Dict]]:
        """Measures retrieval time. Returns (time, few_shots)."""
        start = time.perf_counter()
        emb  = self.nlp_lg(sentence).vector.reshape(1, -1)
        sims = cosine_similarity(emb, train_embeddings).flatten()
        top  = np.argsort(sims)[-n:][::-1]
        few_shots = [
            {"sentence": train_sents[i], "labels": train_labels[i]}
            for i in top
        ]
        end = time.perf_counter()
        return end - start, few_shots

    def count_tokens_breakdown(
        self,
        sentence: str,
        dependency_str: str,
        few_shot_examples: List[Dict],
        instruction: str
    ) -> Dict[str, int]:
        """
        Returns token count broken down by component:
        - dependency: tokens used by the dependency tree
        - few_shot:   tokens used by the retrieved few-shot example
        - llm_core:   tokens used by instruction + query + overhead
        - total:      sum of all
        """
        # 1. Instruction tokens (system message)
        instruction_tokens = len(self.enc.encode(instruction))

        # 2. Few-shot tokens
        few_shot_text = ""
        if few_shot_examples:
            few_shot_text = "### Examples\nHere are some examples to guide you:\n"
            for ex in few_shot_examples:
                few_shot_text += (
                    f"\nSentence: {ex['sentence']}\n"
                    f"Labels: {str(ex['labels'])}\n"
                )
            few_shot_text += "\n### Now, based on instruction and these examples, perform the task for the following sentence:\n"
        few_shot_tokens = len(self.enc.encode(few_shot_text))

        # 3. Dependency tree tokens
        dep_tokens = len(self.enc.encode(f"Dependency Tree:\n{dependency_str}\n"))

        # 4. Query + instruction + overhead tokens
        query_text   = f"Query: {sentence}\nLABELS:"
        query_tokens = len(self.enc.encode(query_text))
        overhead     = 8  # OpenAI message format overhead

        # LLM core = instruction + query + overhead (what the LLM "sees" beyond context)
        llm_core_tokens = instruction_tokens + query_tokens + overhead

        total = instruction_tokens + few_shot_tokens + dep_tokens + query_tokens + overhead

        return {
            "dependency": dep_tokens,
            "few_shot":   few_shot_tokens,
            "llm_core":   llm_core_tokens,
            "total":      total
        }

    def time_llm_prediction(
        self,
        sentence: str,
        dependency_str: str,
        few_shot_examples: List[Dict],
        instruction: str
    ) -> float:
        """Measures only the LLM API call time in seconds."""
        few_shot_text = ""
        if few_shot_examples:
            few_shot_text = "### Examples\nHere are some examples to guide you:\n"
            for ex in few_shot_examples:
                few_shot_text += (
                    f"\nSentence: {ex['sentence']}\n"
                    f"Labels: {str(ex['labels'])}\n"
                )
            few_shot_text += "\n### Now, based on instruction and these examples, perform the task for the following sentence:\n"

        user_content  = f"{few_shot_text}Query: {sentence}\n"
        user_content += f"Dependency Tree:\n{dependency_str}\n"
        user_content += "LABELS:"

        messages = [
            {"role": "system", "content": instruction},
            {"role": "user",   "content": user_content}
        ]

        start = time.perf_counter()
        try:
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                max_tokens=512,
                temperature=0.0
            )
            _ = response.choices[0].message.content
        except Exception as e:
            print(f"  [API Error]: {e}")
        end = time.perf_counter()
        return end - start


# ==========================================
# MAIN TIMING EXPERIMENT
# ==========================================
if __name__ == "__main__":

    # --- Load data ---
    print("Loading data...")
    train_sents, train_labels = read_line_examples_from_file(TRAIN_DATA_PATH)
    test_sents,  test_labels  = read_line_examples_from_file(TEST_DATA_PATH)

    sample_sents = test_sents[:NUM_SAMPLES]
    print(f"Measuring inference time on {len(sample_sents)} sentences.")

    # --- Build retriever index ---
    print("Building retriever index...")
    timer = InferenceTimer(api_key=api_key, base_url=BASE_URL)
    train_embeddings = np.array([
        timer.nlp_lg(s).vector for s in tqdm(train_sents, desc="Encoding train")
    ])

    # --- Instruction ---
    instruction = """
ROLE:
You are an expert ASQP (aspect sentiment quadruple prediction) annotation bot. Your processing must be precise, systematic, and strictly follow all rules.

TASK DEFINITION:
Your sole task is to extract all sentiment quadruples from a user-provided sentence.
Your response MUST be a valid Python list of lists, with no other text or explanations.
The output format for each quadruple is a list of four strings:
[<Aspect Term>, <Aspect Category>, <Sentiment Polarity>, <Opinion Term>]

ELEMENT DEFINITIONS:
1.  Aspect Term: The specific noun or phrase being reviewed.
    - It MUST be a term from the restaurant domain (e.g., "food", "service", "waiter").
    - Use the exact string "NULL" if the opinion is general or the aspect is implicit.
2.  Aspect Category: The general class for the Aspect Term.
    - You MUST choose from this predefined list only:
        - `ambience general`
        - `drinks prices`
        - `drinks quality`
        - `drinks style_options`
        - `food prices`
        - `food quality`
        - `food style_options`
        - `location general`
        - `restaurant general`
        - `restaurant miscellaneous`
        - `restaurant prices`
        - `service general`
3.  Sentiment: The polarity of the opinion.
    - It MUST be one of three strings: "positive", "negative", or "neutral".
4.  Opinion Term: The exact word(s) from the sentence expressing the sentiment.

EXTRACTION RULES:
1.  Concise Spans: The Opinion Term must be the most concise phrase that still captures the full sentiment. Do not include unnecessary words.
2.  Extract All Opinions: You must extract a separate quadruple for every distinct opinion expressed in the sentence.
3.  Implied Sentiment: Analyze comparative and conditional phrases to determine the implied sentiment.

STEP BY STEP PROCESS:
To ensure accuracy, think step-by-step before generating the final list:
1.  First, identify all opinion-expressing words/phrases in the sentence.
2.  Second, for each opinion, identify its specific aspect phrase. If none exist, that aspect is "NULL".
3.  Third, determine the sentiment polarity (positive, negative, neutral) and the correct aspect category from the provided list.
4.  Finally, construct the list of all quadruples according to all rules above.

FINAL OUTPUT INSTRUCTION:
Your final output MUST ONLY be a valid Python list of lists.
- Do not include explanations, reasoning, or markdown formatting.
- Do not include extra text before or after the list.
- Each inner list must contain exactly 4 strings.
"""

    # --- Storage ---
    parsing_times    = []
    retrieval_times  = []
    llm_times        = []
    total_times      = []
    dep_token_counts = []
    fs_token_counts  = []
    llm_token_counts = []
    total_token_counts = []

    print(f"\n--- Running Inference Timing on {len(sample_sents)} samples ---\n")

    for i, sentence in enumerate(tqdm(sample_sents, desc="Timing")):

        # 1 — Time dependency parsing
        t_parse, dependency_str = timer.time_dependency_parsing(sentence)

        # 2 — Time retrieval
        t_retrieve, few_shots = timer.time_retrieval(
            sentence, train_embeddings,
            train_sents, train_labels,
            n=NUM_FEW_SHOTS
        )

        # 3 — Count tokens breakdown (no API call)
        token_breakdown = timer.count_tokens_breakdown(
            sentence, dependency_str,
            few_shots, instruction
        )

        # 4 — Time LLM prediction
        t_llm = timer.time_llm_prediction(
            sentence, dependency_str,
            few_shots, instruction
        )

        t_total = t_parse + t_retrieve + t_llm

        parsing_times.append(t_parse)
        retrieval_times.append(t_retrieve)
        llm_times.append(t_llm)
        total_times.append(t_total)
        dep_token_counts.append(token_breakdown["dependency"])
        fs_token_counts.append(token_breakdown["few_shot"])
        llm_token_counts.append(token_breakdown["llm_core"])
        total_token_counts.append(token_breakdown["total"])

        # API rate limit
        time.sleep(1.0)

        # Print progress every 20 samples
        if (i + 1) % 20 == 0:
            print(f"\n--- Progress {i+1}/{len(sample_sents)} ---")
            print(f"  Parsing avg:        {statistics.mean(parsing_times):.4f}s")
            print(f"  Retrieval avg:      {statistics.mean(retrieval_times):.4f}s")
            print(f"  LLM avg:            {statistics.mean(llm_times):.4f}s")
            print(f"  Total avg:          {statistics.mean(total_times):.4f}s")
            print(f"  Dep tokens avg:     {statistics.mean(dep_token_counts):.1f}")
            print(f"  Few-shot tokens avg:{statistics.mean(fs_token_counts):.1f}")
            print(f"  LLM tokens avg:     {statistics.mean(llm_token_counts):.1f}")
            print(f"  Total tokens avg:   {statistics.mean(total_token_counts):.1f}")

    # --- Final Results ---
    avg_dep_tokens   = round(statistics.mean(dep_token_counts),   1)
    avg_fs_tokens    = round(statistics.mean(fs_token_counts),    1)
    avg_llm_tokens   = round(statistics.mean(llm_token_counts),   1)
    avg_total_tokens = round(statistics.mean(total_token_counts), 1)

    results = {
        "Dependency Parsing": {
            "mean":       statistics.mean(parsing_times),
            "std":        statistics.stdev(parsing_times),
            "avg_tokens": avg_dep_tokens,
            "complexity": r"$\mathcal{O}(1)$"
        },
        "Few-shot Retrieval": {
            "mean":       statistics.mean(retrieval_times),
            "std":        statistics.stdev(retrieval_times),
            "avg_tokens": avg_fs_tokens,
            "complexity": r"$\mathcal{O}(n)$"
        },
        "LLM Prediction": {
            "mean":       statistics.mean(llm_times),
            "std":        statistics.stdev(llm_times),
            "avg_tokens": avg_llm_tokens,
            "complexity": r"$\mathcal{O}(1)$"
        },
        "Total": {
            "mean":       statistics.mean(total_times),
            "std":        statistics.stdev(total_times),
            "avg_tokens": avg_total_tokens,
            "complexity": r"$\mathcal{O}(n)$"
        }
    }

    print("\n\n========== INFERENCE TIMING RESULTS ==========")
    for component, vals in results.items():
        print(
            f"  {component:22s}: "
            f"Mean={vals['mean']:.4f}s  "
            f"Std={vals['std']:.4f}s  "
            f"Avg Tokens={vals['avg_tokens']}  "
            f"Complexity={vals['complexity']}"
        )
    print("===============================================")

    # --- Save results ---
    with open("inference_timing_results.json", "w") as f:
        json.dump(results, f, indent=4)
    print("\nResults saved to 'inference_timing_results.json'")

    # --- Print LaTeX table ---
    print("\n\n===== LATEX TABLE =====\n")

    print(r"\afterpage{%")
    print(r"\begin{table}[t]")
    print(r"\centering")
    print(r"\footnotesize")
    print(r"\setlength{\tabcolsep}{5pt}")
    print(r"\renewcommand{\arraystretch}{1.15}")
    print(r"\resizebox{\columnwidth}{!}{%")
    print(r"\begin{tabular}{lcccc}")
    print(r"\toprule")
    print(r"\textbf{Component} &")
    print(r"\textbf{Time (s)} &")
    print(r"\textbf{Std} &")
    print(r"\textbf{Avg. Tokens} &")
    print(r"\textbf{Complexity} \\")
    print(r"\midrule")

    rows = [
        "Dependency Parsing",
        "Few-shot Retrieval",
        "LLM Prediction",
        "Total",
    ]

    for key in rows:
        val    = results[key]
        mean   = f"{val['mean']:.3f}"
        std    = f"{val['std']:.3f}"
        tokens = val['avg_tokens']
        comp   = val['complexity']

        if key == "Total":
            print(r"\midrule")
            print(
                f"  \\textbf{{{key}}} & "
                f"\\textbf{{{mean}}} & "
                f"\\textbf{{{std}}} & "
                f"\\textbf{{{tokens}}} & "
                f"\\textbf{{{comp}}} \\\\"
            )
        else:
            print(f"  {key} & {mean} & {std} & {tokens} & {comp} \\\\")

    print(r"\bottomrule")
    print(r"\end{tabular}%")
    print(r"}")
    print(r"\caption{Inference efficiency, token usage, and computational "
          r"complexity of DRIFT on Rest15.}")
    print(r"\label{tab:inference-cost}")
    print(r"\end{table}")
    print(r"}")

Loading data...
Measuring inference time on 100 sentences.
Building retriever index...


Encoding train: 100%|██████████| 834/834 [00:08<00:00, 94.03it/s] 



--- Running Inference Timing on 100 samples ---



Timing:  20%|██        | 20/100 [00:52<03:21,  2.52s/it]


--- Progress 20/100 ---
  Parsing avg:        0.0148s
  Retrieval avg:      0.0144s
  LLM avg:            1.5813s
  Total avg:          1.6106s
  Dep tokens avg:     120.0
  Few-shot tokens avg:79.5
  LLM tokens avg:     592.0
  Total tokens avg:   791.5


...


--- Progress 60/100 ---
  Parsing avg:        0.0155s
  Retrieval avg:      0.0143s
  LLM avg:            1.3209s
  Total avg:          1.3507s
  Dep tokens avg:     115.3
  Few-shot tokens avg:83.5
  LLM tokens avg:     591.1
  Total tokens avg:   790.0


Timing:  80%|████████  | 80/100 [03:03<00:43,  2.16s/it]


--- Progress 80/100 ---
  Parsing avg:        0.0157s
  Retrieval avg:      0.0145s
  LLM avg:            1.2575s
  Total avg:          1.2877s
  Dep tokens avg:     112.0
  Few-shot tokens avg:81.5
  LLM tokens avg:     590.6
  Total tokens avg:   784.0


Timing: 100%|██████████| 100/100 [03:53<00:00,  2.34s/it]


--- Progress 100/100 ---
  Parsing avg:        0.0151s
  Retrieval avg:      0.0139s
  LLM avg:            1.3059s
  Total avg:          1.3349s
  Dep tokens avg:     109.0
  Few-shot tokens avg:81.7
  LLM tokens avg:     590.1
  Total tokens avg:   780.8


========== INFERENCE TIMING RESULTS ==========
  Dependency Parsing    : Mean=0.0151s  Std=0.0041s  Avg Tokens=109.0  Complexity=$\mathcal{O}(1)$
  Few-shot Retrieval    : Mean=0.0139s  Std=0.0039s  Avg Tokens=81.7  Complexity=$\mathcal{O}(n)$
  LLM Prediction        : Mean=1.3059s  Std=0.9227s  Avg Tokens=590.1  Complexity=$\mathcal{O}(1)$
  Total                 : Mean=1.3349s  Std=0.9231s  Avg Tokens=780.8  Complexity=$\mathcal{O}(n)$

Results saved to 'inference_timing_results.json'


===== LATEX TABLE =====

\afterpage{%
\begin{table}[t]
\centering
\footnotesize
\setlength{\tabcolsep}{5pt}
\renewcommand{\arraystretch}{1.15}
\resizebox{\columnwidth}{!}{%
\begin{tabular}{lcccc}
\toprule
\textbf{Component} &
\textbf{Time (s)} &
\